# End of week 1 exercise

To demonstrate your familiarity with Groq API, build a tool that takes a technical question,
and responds with an explanation. This exercise also includes a judge model to compare responses.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [16]:
MODEL_GPT = 'openai/gpt-oss-20b'
MODEL_LLAMA = 'meta-llama/llama-4-scout-17b-16e-instruct'
MODEL_JUDGE = 'moonshotai/kimi-k2-instruct-0905'

print(f"GPT Model: {MODEL_GPT}")
print(f"Llama Model: {MODEL_LLAMA}")
print(f"Judge Model: {MODEL_JUDGE}")

GPT Model: openai/gpt-oss-20b
Llama Model: meta-llama/llama-4-scout-17b-16e-instruct
Judge Model: moonshotai/kimi-k2-instruct-0905


In [ ]:
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if not groq_api_key:
    print("No Groq API key was found - please add GROQ_API_KEY to your .env file")
elif not groq_api_key.startswith("gsk_"):
    print("An API key was found, but it doesn't start with 'gsk_' - please check you're using the right Groq API key")
else:
    print("Groq API key found and looks good so far!")

# Configure Groq client
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

Groq API key found and looks good so far!


In [ ]:
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [17]:
system_prompt = """
You are a senior Python developer and technical instructor with a gift for teaching complex concepts simply.

When explaining code:
- Break it down piece by piece like you're teaching a friend
- Use beautiful, well-structured markdown with:
  * Clear headings (##, ###) to organize sections
  * Code blocks with language specification (```python)
  * Bullet points for key takeaways
  * Tables for comparisons when helpful
  * **Bold** and *italic* for emphasis
  * > Blockquotes for important notes or gotchas
  * --- horizontal rules to separate sections
  * Emojis sparingly to add personality (🐍 for Python, 💡 for insights, ⚠️ for warnings)

Structure your explanations:
1. ## Overview - What does this code do in plain English?
2. ## Breaking It Down - Dissect each component
   - Explain the syntax
   - Show equivalent code if helpful
3. ## Why It Works This Way - The reasoning behind it
4. ## Practical Example - A concrete, runnable example
5. ## Common Pitfalls - What to watch out for
6. ## Key Takeaway - One sentence summary

Make it:
- Clear enough for a beginner
- Insightful enough for an experienced dev
- Visually scannable with proper markdown formatting
- Engaging and actually enjoyable to read

Remember: You're not just explaining code, you're teaching someone to think differently about Python.
"""

def create_messages(question):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Explain this Python code:\n\n{question}"}
    ]

In [18]:
def get_explanation(model, question, model_name="Model"):
    """Get explanation from specified model"""
    try:
        response = groq_client.chat.completions.create(
            model=model,
            messages=create_messages(question)
        )
        
        explanation = response.choices[0].message.content
        token_usage = {}
        
        if hasattr(response, 'usage'):
            token_usage = {
                "prompt_tokens": response.usage.prompt_tokens,
                "completion_tokens": response.usage.completion_tokens,
                "total_tokens": response.usage.total_tokens
            }
        
        return {
            "model": model_name,
            "explanation": explanation,
            "token_usage": token_usage
        }
    except Exception as e:
        return {
            "model": model_name,
            "error": str(e),
            "explanation": None
        }

In [19]:

print("Getting explanations from both models...\n")

gpt_result = get_explanation(MODEL_GPT, question, "openai/gpt-oss-20b")
print(f"✅ Got response from {gpt_result['model']}")

llama_result = get_explanation(MODEL_LLAMA, question, "meta-llama/llama-4-scout-17b-16e-instruct")
print(f"✅ Got response from {llama_result['model']}")

print("\n" + "="*80)
print("EXPLANATIONS RECEIVED")
print("="*80)

if gpt_result['explanation']:
    print(f"\n📋 {gpt_result['model']} EXPLANATION:")
    print("-"*50)
    display(Markdown(gpt_result['explanation']))
    if gpt_result['token_usage']:
        print(f"\nTokens: {gpt_result['token_usage']['total_tokens']}")

if llama_result['explanation']:
    print(f"\n📋 {llama_result['model']} EXPLANATION:")
    print("-"*50)
    display(Markdown(llama_result['explanation']))
    if llama_result['token_usage']:
        print(f"\nTokens: {llama_result['token_usage']['total_tokens']}")

Getting explanations from both models...

✅ Got response from openai/gpt-oss-20b
✅ Got response from meta-llama/llama-4-scout-17b-16e-instruct

EXPLANATIONS RECEIVED

📋 openai/gpt-oss-20b EXPLANATION:
--------------------------------------------------


## 🐍 Overview  
The line

```python
yield from {book.get("author") for book in books if book.get("author")}
```

is a **generator expression inside a `yield from` statement**.  
It takes a list (or any iterable) of book dictionaries, extracts the `author` field from each book **that actually has an author**, removes duplicates (because it uses a **set**), and then *yields* each unique author to whatever called the generator.

> Think of it as: *“Give me every distinct author name that appears in this list of books.”*


---  

## 🔧 Breaking It Down

| Element | What it does | Equivalent code | Notes |
|---------|--------------|-----------------|-------|
| `{ … }` | **Set comprehension** – builds a `set` of values. | `set(book.get("author") for book in books if book.get("author"))` | Sets automatically drop duplicates. |
| `book.get("author")` | Safely fetches the value of the `"author"` key; returns `None` if the key is missing. | `book["author"] if "author" in book else None` | `None` is *falsy*, so it will be excluded by the `if` test. |
| `for book in books` | Iterates over every dictionary in `books`. | `for book in books:` | `books` must be an *iterable* of mapping objects. |
| `if book.get("author")` | Filters out books where the author is missing, `None`, or any *falsy* value (`''`, `0`, etc.). | `if book.get("author") is not None:` | Keeps only truthy author strings. |
| `yield from …` | A generator‑function keyword that delegates to another iterable. It will **yield each element of the set one by one**. | `for author in …: yield author` | Requires the surrounding function to be a generator (`def func():` + `yield`). |

**Equivalent, more verbose version**

```python
def authors():
    authors_set = {book.get("author") for book in books if book.get("author")}
    for author in authors_set:
        yield author
```

---

## 💡 Why It Works This Way

| Reason | Explanation |
|--------|-------------|
| **Set removes duplicates** | A set can only contain unique elements, so if multiple books share the same author, that author appears only once in the result. |
| **`get` protects against KeyError** | Using `book.get("author")` means the code won’t crash if a book dict is missing the key. |
| **`if book.get("author")` filters falsy values** | Only truthy author names (non‑empty strings, etc.) are added to the set. |
| **`yield from` is syntactic sugar** | It eliminates the boilerplate `for …: yield …`. It’s perfect for delegating to an iterable inside a generator. |

---

## 🚀 Practical Example

```python
books = [
    {"title": "The Great Gatsby", "author": "F. Scott Fitzgerald"},
    {"title": "1984", "author": "George Orwell"},
    {"title": "Animal Farm", "author": "George Orwell"},
    {"title": "Untitled"},                     # no author
    {"title": "Mystery Book", "author": ""}    # empty author
]

def unique_authors():
    yield from {book.get("author") for book in books if book.get("author")}

# Consume the generator
for author in unique_authors():
    print(author)
```

**Possible output**

```
F. Scott Fitzgerald
George Orwell
```

> ⚠️ The order is *not guaranteed* because a `set` is unordered.  
> If you need a stable order, use a list comprehension or a generator expression:

```python
yield from (book["author"] for book in books if "author" in book)
```

---

## ❗ Common Pitfalls

| Pitfall | Why it happens | Fix |
|---------|----------------|-----|
| **Using `yield from` outside a generator** | `yield from` can only be used inside a function that contains at least one `yield`. | Make sure the surrounding function has a `yield` or is a generator. |
| **Unexpected `None` values** | `book.get("author")` returns `None` for missing keys, which is falsy and filtered out, but if you *don’t* filter, you’d get `None` in the set. | Keep the `if book.get("author")` guard or explicitly filter `None`. |
| **Unordered results** | Sets don’t preserve insertion order. | Use a list comprehension or a generator expression if order matters. |
| **Empty string treated as missing** | `if book.get("author")` considers `""` falsy, so empty author names are ignored. | If you want to keep empty strings, change the condition to `if book.get("author") is not None`. |

---

## 🎯 Key Takeaway  
`yield from {book.get("author") for book in books if book.get("author")}` is a concise, generator‑friendly way to emit *unique*, *non‑empty* author names from a collection of book dictionaries, gracefully handling missing keys and eliminating duplicates in one line.


Tokens: 2113

📋 meta-llama/llama-4-scout-17b-16e-instruct EXPLANATION:
--------------------------------------------------


## Overview
The given Python code uses a combination of a generator and a set comprehension to yield unique authors from a list of books. 

```python
yield from {book.get("author") for book in books if book.get("author")}
```

## Breaking It Down
Let's dissect this code piece by piece:

### Understanding the Components

* `book.get("author")`: This retrieves the value associated with the key `"author"` from the `book` dictionary. If the key doesn't exist, it returns `None` by default.
* `{...}`: This is a set comprehension, which creates a set of unique values.
* `for book in books`: This iterates over each dictionary in the `books` list.
* `if book.get("author")`: This filters out books without an author (i.e., `book.get("author")` returns `None` or an empty string).
* `yield from`: This is a syntax introduced in Python 3.3, allowing a generator to delegate part of its operations to a sub-generator.

### Equivalent Code

To better understand the code, here's an equivalent version using a traditional loop:

```python
unique_authors = set()
for book in books:
    author = book.get("author")
    if author:
        unique_authors.add(author)

yield from unique_authors
```

## Why It Works This Way
The code uses a set to automatically eliminate duplicate authors. It then uses `yield from` to produce a generator that yields each unique author. This approach has several benefits:

* **Memory Efficiency**: By using a generator, the code avoids loading all authors into memory at once.
* **Unique Authors**: The set comprehension ensures that each author is only yielded once.

## Practical Example
Here's a complete example:

```python
def get_unique_authors(books):
    """Yield unique authors from a list of books."""
    yield from {book.get("author") for book in books if book.get("author")}

# Example usage:
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},
    {"title": "Book 4"}  # No author
]

unique_authors = list(get_unique_authors(books))
print(unique_authors)  # Output: ['Author A', 'Author B']
```

## Common Pitfalls
When using this code, be aware of:

* **None or Missing Authors**: If a book is missing an author or the author is `None`, it will be skipped.
* **Empty Strings as Authors**: If an author's name is an empty string, it will also be skipped.

## Key Takeaway
This Python code efficiently yields unique authors from a list of books using a generator and set comprehension.


Tokens: 930


In [21]:
judge_system_prompt = """
You are an impartial judge evaluating explanations of Python code.
You will be given a technical question and two explanations from different AI models.
Your task is to compare these explanations and determine which one is better.

Evaluate based on:
1. Accuracy - Does it correctly explain what the code does?
2. Clarity - Is it easy to understand?
3. Completeness - Does it cover all important aspects?
4. Examples - Does it provide helpful examples?
5. Teaching quality - Would it help someone learn?

Respond in JSON format with:
- winner: which model won ("gpt", "llama", or "tie")
- reasoning: detailed explanation of your decision
- scores: object with scores for each model (1-10)
- key_differences: what made the difference
"""

def create_judge_messages(question, gpt_explanation, llama_explanation):
    user_content = f"""
QUESTION TO EXPLAIN:
{question}

EXPLANATION FROM openai/gpt-oss-20b:
{gpt_explanation}

EXPLANATION FROM meta-llama/llama-4-scout-17b-16e-instruct:
{llama_explanation}

Please evaluate these explanations and tell me which one is better and why.
"""
    
    return [
        {"role": "system", "content": judge_system_prompt},
        {"role": "user", "content": user_content}
    ]

In [ ]:
print("\n" + "="*80)
print("🧑‍⚖️ JUDGE MODEL EVALUATION")
print("="*80)

if gpt_result['explanation'] and llama_result['explanation']:
    try:
        judge_response = groq_client.chat.completions.create(
            model=MODEL_JUDGE,
            messages=create_judge_messages(question, gpt_result['explanation'], llama_result['explanation']),
            response_format={"type": "json_object"}
        )
        
        judge_result = json.loads(judge_response.choices[0].message.content)
        
        print(f"\n🏆 WINNER: {judge_result.get('winner', 'unknown').upper()}")
        
        if 'scores' in judge_result:
            print("\n📊 SCORES:")
            for model, score in judge_result['scores'].items():
                print(f"   {model}: {score}/10")
        
        print(f"\n💭 REASONING:")
        print(judge_result.get('reasoning', 'No reasoning provided'))
        
        if 'key_differences' in judge_result:
            print(f"\n🔑 KEY DIFFERENCES:")
            print(judge_result['key_differences'])
        
        if hasattr(judge_response, 'usage'):
            print(f"\n📈 Judge token usage: {judge_response.usage.total_tokens}")
            
    except Exception as e:
        print(f"❌ Error during judgment: {e}")
else:
        print("❌ Cannot compare - one or both explanations failed")


🧑‍⚖️ JUDGE MODEL EVALUATION

🏆 WINNER: GPT

📊 SCORES:
   gpt: 9/10
   llama: 6/10

💭 REASONING:
GPT’s explanation is superior across all criteria. It is more accurate (correctly noting that the set is built eagerly before the generator starts), clearer (uses a side-by-side table and explicit equivalences), and more complete (covers pitfalls, ordering issues, and alternative approaches). The practical example is richer and the teaching aids (emoji, call-out boxes, extra code snippets) make it easier to learn from. Llama’s version is shorter and omits critical details like set ordering, the fact that the set is fully materialized before yielding, and how falsy values are treated.

🔑 KEY DIFFERENCES:
['Accuracy: GPT explicitly states the set is constructed first, Llama implies streaming behavior.', 'Clarity: GPT’s table and verbose equivalent code are clearer than Llama’s short paragraph.', 'Completeness: GPT covers ordering, falsy vs None, and common pitfalls; Llama only briefly mention

In [24]:
# Cell 10: Streaming version for single model
def stream_explanation(model, question, model_name="Model"):
    """Stream explanation from a model"""
    print(f"\n{'='*60}")
    print(f"🤖 {model_name} EXPLANATION (streaming)")
    print(f"{'='*60}\n")
    
    try:
        stream = groq_client.chat.completions.create(
            model=model,
            messages=create_messages(question),
            stream=True
        )
        
        response = ""
        display_handle = display(Markdown(""), display_id=True)
        
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                response += chunk.choices[0].delta.content
                update_display(Markdown(response), display_id=display_handle.display_id)
        
        print("\n✅ Explanation complete!")
        
    except Exception as e:
        print(f"❌ Error: {e}")

# Test streaming with one model (optional)
stream_explanation(MODEL_GPT, question, "openai/gpt-oss-20b")


🤖 openai/gpt-oss-20b EXPLANATION (streaming)



## Overview  
🐍 **What the line does**  
The expression

```python
yield from {book.get("author") for book in books if book.get("author")}
```

is typically written inside a *generator function*.  
It:

1. **Builds a set** of all author names found in the `books` iterable.  
2. **Filters out** any entries where an author is missing, `None`, or otherwise falsy.  
3. **Yields each author** (as a separate value) back to the caller.  

In plain English: *“Give me every unique author that appears in `books`, but only the ones that actually exist.”*

---

## Breaking It Down  
Let’s dissect each part step by step.

| Piece | Explanation | Equivalent (expanded) |
|-------|-------------|------------------------|
| `yield from` | Delegates to an *iterator*. It automatically iterates over the provided iterable, yielding each of its items. | ```python<br>for item in iterable:<br>    yield item<br>``` |
| `{book.get("author") for book in books if book.get("author")}` | **Set comprehension**: Creates a set of values. | ```python<br>authors = set()<br>for book in books:<br>    author = book.get("author")<br>    if author:<br>        authors.add(author)<br>``` |
| `book.get("author")` | Safely fetches the value of the `"author"` key; returns `None` if the key is missing. | ```python<br>author = book["author"] if "author" in book else None<br>``` |

> ⚠️ **Important**:  
> `yield from` can only appear *inside* a generator function (`def foo(): yield ...`). Placing it in a normal function will raise a `SyntaxError`.

---

## Why It Works This Way  
### 1. Set comprehension for uniqueness  
- A `set` automatically discards duplicate values.  
- By putting the expression inside curly braces `{}`, we’re telling Python to collect all author names into a set.

### 2. The `if book.get("author")` guard  
- `dict.get` returns `None` when the key is missing.  
- In a Boolean context, `None`, `False`, `0`, `""`, and empty containers evaluate to `False`.  
- The guard ensures we only include truthy author values.

### 3. Delegation with `yield from`  
- `yield from` is syntactic sugar for a loop that yields each item of an iterable.  
- It simplifies code when you want to “hand over” all values of another generator or iterator.

---

## Practical Example  
Below is a self‑contained, runnable snippet demonstrating the code in action.

```python
# books.py
def unique_authors(books):
    """Yield every unique author present in the books list."""
    yield from {book.get("author") for book in books if book.get("author")}

# Sample data
books = [
    {"title": "Python 101", "author": "Alice"},
    {"title": "Advanced Python", "author": "Bob"},
    {"title": "Python 101", "author": "Alice"},
    {"title": "Learning AI", "author": None},
    {"title": "Data Science", "author": "Charlie"},
]

# Consume the generator
for author in unique_authors(books):
    print(author)
```

**Output**

```
Alice
Bob
Charlie
```

> 💡 *Note*: The order is not guaranteed because sets are unordered. If order matters, consider using `dict.fromkeys()` or `OrderedSet` from `more-itertools`.

---

## Common Pitfalls  
| Pitfall | Why it matters | How to avoid |
|---------|----------------|--------------|
| **Using `yield from` outside a generator** | SyntaxError: `yield from` is only valid inside a generator. | Wrap the code in `def func():` and ensure the function is called as a generator. |
| **Relying on order** | Sets do not preserve insertion order. | Use an ordered container or collect into a list if order matters. |
| **Assuming `yield from` returns a list** | It yields items one at a time, not a collection. | Convert to a list if you need a collection: `authors = list(unique_authors(books))`. |
| **Missing falsy values** | `if book.get("author")` will skip authors that are empty strings `""` or `0`. | Adjust the guard if you want to include those values: `if "author" in book`. |

---

## Key Takeaway  
> **`yield from` delegates to the set of unique, truthy authors extracted from `books`, allowing the generator to stream each author value one at a time.** 🌟


✅ Explanation complete!


In [29]:

def ask_and_judge(new_question):
    """Ask a new question and get judged comparison"""
    print("\n" + "="*80)
    print(f"❓ NEW QUESTION: {new_question[:100]}..." if len(new_question) > 100 else f"❓ NEW QUESTION: {new_question}")
    print("="*80)
    
    # Get explanations
    gpt = get_explanation(MODEL_GPT, new_question, "openai/gpt-oss-20b")
    llama = get_explanation(MODEL_LLAMA, new_question, "meta-llama/llama-4-scout-17b-16e-instruct")
    
    if gpt['explanation'] and llama['explanation']:
        # Judge them
        judge_response = groq_client.chat.completions.create(
            model=MODEL_JUDGE,
            messages=create_judge_messages(new_question, gpt['explanation'], llama['explanation']),
            response_format={"type": "json_object"}
        )
        
        result = json.loads(judge_response.choices[0].message.content)
        
        print(f"\n WINNER: {result.get('winner', 'unknown').upper()}")
        print(f"\n {result.get('reasoning', '')[:300]}...")
        
        return result
    else:
        print(" Failed to get explanations")
        return None

# Example: Ask your own question
my_question = "Explain how Python decorators work with an example"
ask_and_judge(my_question)


❓ NEW QUESTION: Explain how Python decorators work with an example

 WINNER: GPT

 The GPT explanation is superior in almost every dimension. It starts with an intuitive metaphor ('wrapper') and immediately shows the anatomy of a decorator in a clear table. It then walks through the execution flow step-by-step, explicitly explaining that the original function is replaced by the re...


{'winner': 'gpt',
 'reasoning': "The GPT explanation is superior in almost every dimension. It starts with an intuitive metaphor ('wrapper') and immediately shows the anatomy of a decorator in a clear table. It then walks through the execution flow step-by-step, explicitly explaining that the original function is replaced by the returned wrapper. The practical example is far more comprehensive: it demonstrates three different real-world decorators (logging, timing, caching) and shows how they compose, including the crucial point that order matters. It also covers advanced details like functools.wraps, *args/**kwargs for signature transparency, and common pitfalls with specific fixes. The Llama explanation is accurate but minimal; it only shows a single trivial before/after print decorator and a basic timing example. It lacks depth, multiple use-cases, and the rich pedagogical structure (tables, pitfalls list, takeaway summary) provided by GPT. GPT’s answer would leave a reader ready to

In [28]:

different_question = """
Explain what this does and why:
def memoize(func):
    cache = {}
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    return wrapper
"""



# Get explanations for the new question
gpt2 = get_explanation(MODEL_GPT, different_question, "openai/gpt-oss-20b")
llama2 = get_explanation(MODEL_LLAMA, different_question, "meta-llama/llama-4-scout-17b-16e-instruct")

if gpt2['explanation'] and llama2['explanation']:
    print("\n First 200 chars of each explanation:")
    print(f"\nGPT-OSS-20b: {gpt2['explanation'][:200]}...")
    print(f"\nLLAMA-4-SCOUT: {llama2['explanation'][:200]}...")
    
    # Use judge to compare
    judge2 = groq_client.chat.completions.create(
        model=MODEL_JUDGE,
        messages=create_judge_messages(different_question, gpt2['explanation'], llama2['explanation']),
        response_format={"type": "json_object"}
    )
    
    result2 = json.loads(judge2.choices[0].message.content)
    print(f"\n WINNER: {result2.get('winner', 'unknown').upper()}")
else:
    print(" Failed to get explanations")


 First 200 chars of each explanation:

GPT-OSS-20b: ## 🐍 Overview  
This snippet implements a **simple memoization decorator**.  
It remembers the results of a function call so that future calls with the same arguments return the cached value instantly...

LLAMA-4-SCOUT: ## Overview
The given Python code implements a decorator function called `memoize`. This decorator is used to **cache the results of expensive function calls** and reuse them when the same inputs occu...

 WINNER: GPT
